In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("SILVER LAYER: Data Cleaning & Validation")
print("="*60)

# Read from Bronze
path = "abfss://8b73c65d-76d6-466c-96b4-ed517828198f@onelake.dfs.fabric.microsoft.com/bffa460f-5aa4-4fec-ad78-8bb5cf9b4505/Tables/dbo/bronze_crashes_raw"
df_bronze = spark.read.format("delta").load(path)

#df_bronze = spark.table("bronze_crashes_raw")
initial_count = df_bronze.count()
print(f"Loaded from Bronze: {initial_count:,} records")

# Copy to working dataframe
df_silver = df_bronze

print("\nData loaded successfully")


StatementMeta(, ae5d0fe9-df21-4677-bcfc-9088fa59a6c7, 3, Finished, Available, Finished, False)

SILVER LAYER: Data Cleaning & Validation
Loaded from Bronze: 120,579 records

Data loaded successfully


In [2]:
print("\nStep 1: Removing duplicates...")

# Count before
before_dedup = df_silver.count()

# Remove duplicates based on objectId (unique crash identifier)
df_silver = df_silver.dropDuplicates(["objectId"])

# Count after
after_dedup = df_silver.count()
duplicates_removed = before_dedup - after_dedup

print(f"  - Before: {before_dedup:,} records")
print(f"  - After: {after_dedup:,} records")
print(f"  - Duplicates removed: {duplicates_removed:,}")

if duplicates_removed == 0:
    print("No duplicates found - data quality good!")
else:
    print(f"Removed {duplicates_removed:,} duplicate records")

StatementMeta(, ae5d0fe9-df21-4677-bcfc-9088fa59a6c7, 4, Finished, Available, Finished, False)


Step 1: Removing duplicates...
  - Before: 120,579 records
  - After: 120,579 records
  - Duplicates removed: 0
No duplicates found - data quality good!


In [3]:
print("\nStep 2: Handling missing coordinates...")

# Count nulls
null_x = df_silver.filter(F.col("X").isNull()).count()
null_y = df_silver.filter(F.col("Y").isNull()).count()

print(f"  - Missing X (longitude): {null_x:,}")
print(f"  - Missing Y (latitude): {null_y:,}")

# Remove records with missing coordinates (can't analyze without location)
before_filter = df_silver.count()
df_silver = df_silver.filter(
    (F.col("X").isNotNull()) & 
    (F.col("Y").isNotNull())
)
after_filter = df_silver.count()

print(f"  - Removed: {before_filter - after_filter:,} records")
print("All records now have valid coordinates")

StatementMeta(, ae5d0fe9-df21-4677-bcfc-9088fa59a6c7, 5, Finished, Available, Finished, False)


Step 2: Handling missing coordinates...
  - Missing X (longitude): 0
  - Missing Y (latitude): 0
  - Removed: 0 records
All records now have valid coordinates


In [4]:
print("\nStep 3: Validating geographic bounds...")

# Auckland boundaries
AUCKLAND_LON_MIN = 174.45
AUCKLAND_LON_MAX = 175.25
AUCKLAND_LAT_MIN = -37.15
AUCKLAND_LAT_MAX = -36.65

# Count out of bounds
out_of_bounds = df_silver.filter(
    ~(F.col("X").between(AUCKLAND_LON_MIN, AUCKLAND_LON_MAX)) |
    ~(F.col("Y").between(AUCKLAND_LAT_MIN, AUCKLAND_LAT_MAX))
).count()

print(f"  - Records outside Auckland bounds: {out_of_bounds:,}")

# Filter to Auckland only
before_geo = df_silver.count()
df_silver = df_silver.filter(
    (F.col("X").between(AUCKLAND_LON_MIN, AUCKLAND_LON_MAX)) &
    (F.col("Y").between(AUCKLAND_LAT_MIN, AUCKLAND_LAT_MAX))
)
after_geo = df_silver.count()

print(f"  - Filtered to Auckland region: {after_geo:,} records")
print(f"  - Removed {before_geo - after_geo:,} out-of-bounds records")
print("All records within Auckland geographic bounds")

StatementMeta(, ae5d0fe9-df21-4677-bcfc-9088fa59a6c7, 6, Finished, Available, Finished, False)


Step 3: Validating geographic bounds...
  - Records outside Auckland bounds: 8,922
  - Filtered to Auckland region: 111,657 records
  - Removed 8,922 out-of-bounds records
All records within Auckland geographic bounds


In [5]:
print("\nStep 4: Standardizing severity categories...")

# Show original distribution
print("  Original severity values:")
df_silver.groupBy("crashSeverity").count().orderBy(F.desc("count")).show()

# Standardize severity names
df_silver = df_silver.withColumn(
    "crashSeverity_clean",
    F.when(F.col("crashSeverity") == "Fatal Crash", "Fatal")
     .when(F.col("crashSeverity") == "Serious Crash", "Serious")
     .when(F.col("crashSeverity") == "Minor Crash", "Minor")
     .when(F.col("crashSeverity") == "Non Injury Crash", "Non-Injury")
     .otherwise("Unknown")
)

# Show cleaned distribution
print("\n  Cleaned severity distribution:")
df_silver.groupBy("crashSeverity_clean").count().orderBy(F.desc("count")).show()

print("Severity categories standardized")

StatementMeta(, ae5d0fe9-df21-4677-bcfc-9088fa59a6c7, 7, Finished, Available, Finished, False)


Step 4: Standardizing severity categories...
  Original severity values:
+----------------+-----+
|   crashSeverity|count|
+----------------+-----+
|Non-Injury Crash|80243|
|     Minor Crash|26051|
|   Serious Crash| 4986|
|     Fatal Crash|  377|
+----------------+-----+


  Cleaned severity distribution:
+-------------------+-----+
|crashSeverity_clean|count|
+-------------------+-----+
|            Unknown|80243|
|              Minor|26051|
|            Serious| 4986|
|              Fatal|  377|
+-------------------+-----+

Severity categories standardized


In [6]:
print("\nStep 5: Fixing data types...")

# Convert year to integer
df_silver = df_silver.withColumn(
    "crashYear_int", 
    F.col("crashYear").cast("integer")
)

# Validate year range
df_silver = df_silver.filter(
    F.col("crashYear_int").between(2015, 2025)
)

# Convert numeric fields
numeric_cols = ["speedLimit", "NumberOfLanes", "crashSHour"]

for col_name in numeric_cols:
    if col_name in df_silver.columns:
        df_silver = df_silver.withColumn(
            col_name,
            F.col(col_name).cast("integer")
        )

print(f"  Data types standardized")
print(f"  - Years: 2015-2025")
print(f"  - Numeric fields converted: {numeric_cols}")

StatementMeta(, ae5d0fe9-df21-4677-bcfc-9088fa59a6c7, 8, Finished, Available, Finished, False)


Step 5: Fixing data types...
  Data types standardized
  - Years: 2015-2025
  - Numeric fields converted: ['speedLimit', 'NumberOfLanes', 'crashSHour']


In [7]:
print("\nStep 6: Creating user type flags...")

# Create binary flags for user types
user_types = {
    "bicycle": "has_bicycle",
    "pedestrian": "has_pedestrian",
    "motorcycle": "has_motorcycle",
    "moped": "has_moped",
    "truck": "has_truck",
    "bus": "has_bus",
    "taxi": "has_taxi",
    "schoolBus": "has_schoolBus"
}

# for original_col, new_col in user_types.items():
#     if original_col in df_silver.columns:
#         df_silver = df_silver.withColumn(
#             new_col,
#             F.when(F.col(original_col) > 0, 1).otherwise(0)
#         )

# coalesce handles Blank (null) values
for original_col, new_col in user_types.items():
    if original_col in df_silver.columns:
        df_silver = df_silver.withColumn(
            new_col,
            F.when(F.coalesce(F.col(original_col), F.lit(0)) > 0, 1).otherwise(0)
        )

# State highway flag — from crashSHDescription (NOT crashSHour)
df_silver = df_silver.withColumn(
    "is_state_highway",
    F.when(F.col("crashSHDescription") == "Yes", 1).otherwise(0)
)

# Holiday flag — handle empty strings too
df_silver = df_silver.withColumn(
    "is_holiday",
    F.when(
        F.col("holiday").isNotNull() & (F.col("holiday") != ""), 1
    ).otherwise(0)
)

# NEW — standardise additional columns

# Road gradient (replaces missing roadCurvature)
df_silver = df_silver.withColumn(
    "flatHill_clean",
    F.when(F.col("flatHill") == "Flat", "Flat")
     .when(F.col("flatHill") == "Hill Road", "Hill Road")
     .otherwise("Unknown")
)

# Road lane configuration
df_silver = df_silver.withColumn(
    "roadLane_clean",
    F.when(F.col("roadLane").isin(["2 way", "1 way", "Off road"]), F.col("roadLane"))
     .otherwise("Unknown")
)

# Urban/Open classification
df_silver = df_silver.withColumn(
    "urban_clean",
    F.when(F.col("urban").isin(["Urban", "Open"]), F.col("urban"))
     .otherwise("Unknown")
)

# Secondary weather condition
df_silver = df_silver.withColumn(
    "weatherB_clean",
    F.when(F.col("weatherB").isin(["Frost", "Strong Wind", "None"]), F.col("weatherB"))
     .otherwise("Unknown")
)

# Injury count columns — cast integers, fill nulls
for cnt_col in ["fatalCount", "seriousInjuryCount", "minorInjuryCount"]:
    df_silver = df_silver.withColumn(
        cnt_col, F.coalesce(F.col(cnt_col), F.lit(0)).cast("integer")
    )

# Create vulnerable user flag
df_silver = df_silver.withColumn(
    "vulnerable_user_involved",
    F.when(
        (F.col("has_bicycle") == 1) | 
        (F.col("has_pedestrian") == 1) | 
        (F.col("has_motorcycle") == 1) |
        (F.col("has_moped") == 1),
        1
    ).otherwise(0)
)

# Create heavy vehicle flag
df_silver = df_silver.withColumn(
    "heavy_vehicle_involved",
    F.when(
        (F.col("has_truck") == 1) | 
        (F.col("has_bus") == 1),
        1
    ).otherwise(0)
)

print(f"  User type flags created: {len(user_types)} types")
print(f"  Derived flags: vulnerable_user_involved, heavy_vehicle_involved")

# Show distribution
print("\n  Vulnerable user involvement:")
df_silver.groupBy("vulnerable_user_involved").count().show()

StatementMeta(, ae5d0fe9-df21-4677-bcfc-9088fa59a6c7, 9, Finished, Available, Finished, False)


Step 6: Creating user type flags...
  User type flags created: 8 types
  Derived flags: vulnerable_user_involved, heavy_vehicle_involved

  Vulnerable user involvement:
+------------------------+-----+
|vulnerable_user_involved|count|
+------------------------+-----+
|                       1|12210|
|                       0|99447|
+------------------------+-----+



In [8]:
print("\nStep 7: Final validation and save...")

# Final record count
final_count = df_silver.count()

# Summary statistics
print("\nSILVER LAYER SUMMARY")
print("="*60)
print(f"Input (Bronze): {initial_count:,} records")
print(f"Output (Silver): {final_count:,} records")
print(f"Records removed: {initial_count - final_count:,} ({((initial_count - final_count)/initial_count*100):.1f}%)")

# Data quality metrics
print("\nData Quality Metrics:")
print(f"  - Null coordinates: 0")
print(f"  - Out of bounds: 0")
print(f"  - Invalid years: 0")
print(f"  - Duplicate crashes: 0")

# Write to Delta table
print("\nWriting Silver table...")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_crashes_cleaned")

print("Silver table created: silver_crashes_cleaned")

# Verify
verify_count = spark.table("silver_crashes_cleaned").count()
print(f"Verified: {verify_count:,} records")

print("\nSILVER LAYER COMPLETE!")

StatementMeta(, ae5d0fe9-df21-4677-bcfc-9088fa59a6c7, 10, Finished, Available, Finished, False)


Step 7: Final validation and save...

SILVER LAYER SUMMARY
Input (Bronze): 120,579 records
Output (Silver): 111,657 records
Records removed: 8,922 (7.4%)

Data Quality Metrics:
  - Null coordinates: 0
  - Out of bounds: 0
  - Invalid years: 0
  - Duplicate crashes: 0

Writing Silver table...
Silver table created: silver_crashes_cleaned
Verified: 111,657 records

SILVER LAYER COMPLETE!
